Check DP2 deep coadd images.

Based on DP2 tutorials 103.5, 103.7, 202.1, Padma's notebook.

Please run it on the [RSP](https://data.lsst.cloud). 

In [ ]:
from lsst.daf.butler import Butler
import lsst.afw.display as afw_display
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth

from pyvo.dal.adhoc import SodaQuery

import io
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import AsinhStretch, ImageNormalize, make_lupton_rgb
from astropy import units as u

%matplotlib inline

In [ ]:
def get_dl_result(band, ra, dec, radius):

    circle = (ra, dec, radius)

    results = sia_client.search(pos=circle, calib_level=3,
                                dpsubtype='lsst.deep_coadd')
    print("Num of sia_client search result: ", len(results))

    band_arr = results['lsst_band']

    index = int(np.where(band_arr == band)[0][0])

    dl_result = discovery.get_datalink_results(results[index])
    
    print(f"Datalink status: {dl_result.status}.")

    return dl_result
    

def get_cutout(band, ra, dec, radius=0.01):

    dl_result = get_dl_result(band, ra, dec, radius)

    sq = SodaQuery.from_resource(dl_result,
                                 dl_result.get_adhocservice_by_id("cutout-sync"),
                                 session=get_pyvo_auth())

    sq.circle = (ra * u.deg, dec * u.deg, radius * u.deg)

    cutout_bytes = sq.execute_stream().read()
    sq.raise_if_error()

    cutout = read_archive(io.BytesIO(cutout_bytes))

    return cutout

In [ ]:
def normalize_band(image, asinh_a=0.002):
    
    data = image.array

    vmin = -0.03
    vmax = 200

    norm = ImageNormalize(vmin=vmin, vmax=vmax,
                          stretch=AsinhStretch(a=asinh_a),
                          clip=True,
                         )

    scaled = norm(data)
    return scaled


def combine_RGB(R_image, G_image, B_image):
    
    R_channel = normalize_band(R_image)
    G_channel = normalize_band(G_image)
    B_channel = normalize_band(B_image)

    RGB_image = np.dstack([R_channel, G_channel, B_channel])

    return RGB_image

    

In [ ]:
def get_coadd(band='r', ra=53.076, dec=-28.110):

    where = "band.name = :band AND patch.region OVERLAPS POINT(:ra, :dec)"
    bind={"band": band, "ra": ra, "dec": dec}
    dataset_refs = butler.query_datasets("deep_coadd", where=where, bind=bind)

    print("Num of images found: ", len(dataset_refs))

    ref = dataset_refs[0]
    deep_coadd = butler.get(ref)

    pretty_coadd = deep_coadd.copy()
    pretty_coadd.apply_background('pretty')

    return deep_coadd, pretty_coadd

In [ ]:
def plot_afw(image):

    fig, ax = plt.subplots(figsize=(6,6))
    display = afw_display.Display(frame=fig)
    #display.scale('linear', 'zscale')
    display.scale('linear', -75, 125)
    display.image(image)


In [ ]:
def plot_RGB(R_image, G_image, B_image, make_lupton=False, lim=None):

    RGB_image = combine_RGB(R_image, G_image, B_image)
    if make_lupton:
        RGB_image = make_lupton_rgb(R_image.array,
                                    G_image.array,
                                    B_image.array,
                                    stretch=0.002, Q=0.001)

    if lim is not None:
        xmin, xmax, ymin, ymax = lim
        RGB_image = RGB_image[xmin:xmax, ymin:ymax]
    
    fig = plt.figure(figsize=(6,6))
    im = plt.imshow(RGB_image,
                    origin='lower')

    

In [ ]:
afw_display.setDefaultBackend("matplotlib")
butler = Butler.from_config('dp2', collections='dp2')

discovery = RSPDiscovery("dp2")
sia_client = discovery.get_sia_client()

In [ ]:
_, pretty_coadd_i = get_coadd('i')
_, pretty_coadd_r = get_coadd('r')
_, pretty_coadd_g = get_coadd('g')

In [ ]:
#pretty_coadd_i.bbox.x

In [ ]:
plot_afw(pretty_coadd_r.image)

In [ ]:
plot_RGB(pretty_coadd_i.image, pretty_coadd_r.image, pretty_coadd_g.image)
plot_RGB(pretty_coadd_i.image, pretty_coadd_r.image, pretty_coadd_g.image, True)

In [ ]:
plot_RGB(pretty_coadd_i.image, pretty_coadd_r.image, pretty_coadd_g.image, False, [0,300,0,300])
plot_RGB(pretty_coadd_i.image, pretty_coadd_r.image, pretty_coadd_g.image, True, [0,300,0,300])

In [ ]:
target_ra, target_dec = 52.98, -28.13
band = 'r'

In [ ]:
cutout = get_cutout(band, target_ra, target_dec)
print(cutout)

In [ ]:
plot_afw(cutout)

In [ ]:
cutout_g = get_cutout('g', target_ra, target_dec)
cutout_r = get_cutout('r', target_ra, target_dec)
cutout_i = get_cutout('i', target_ra, target_dec)

In [ ]:
plot_RGB(cutout_i, cutout_r, cutout_g)
plot_RGB(cutout_i, cutout_r, cutout_g, True)